In [1]:
import os
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("SUPABASE_URL või SUPABASE_KEY puudub .env failist.")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

os.makedirs("output", exist_ok=True)


def get_data(table_name):
    data = []
    page_size = 1000
    page = 0

    while True:
        response = (
            supabase
            .table(table_name)
            .select("*")
            .range(page * page_size, (page + 1) * page_size - 1)
            .execute()
        )

        rows = response.data if response.data else []
        data.extend(rows)

        if len(rows) < page_size:
            break

        page += 1

    return pd.DataFrame(data)


def safe_qcut(series, q, labels, reverse=False):
    s = series.rank(method="first") if series.nunique() < q else series

    if reverse:
        s = -s

    try:
        return pd.qcut(s, q=q, labels=labels, duplicates="drop")
    except ValueError:
        ranked = series.rank(method="first", ascending=not reverse)
        return pd.qcut(ranked, q=q, labels=labels)


# andmete laadimine
sales_df = get_data("sales")
customers_df = get_data("customers")

print("Müügiandmete kuju:", sales_df.shape)
print("Klientide andmete kuju:", customers_df.shape)
print("Müügiandmete veerud:", sales_df.columns.tolist())
print("Klientide andmete veerud:", customers_df.columns.tolist())

# kontroll: vajalikud veerud
required_sales_cols = ["invoice_id", "customer_id", "sale_date", "total_price", "sale_id"]
required_customer_cols = ["customer_id", "first_name", "last_name", "email", "phone", "city"]

missing_sales = [col for col in required_sales_cols if col not in sales_df.columns]
missing_customers = [col for col in required_customer_cols if col not in customers_df.columns]

if missing_sales:
    raise ValueError(f"Sales tabelist puuduvad veerud: {missing_sales}")

if missing_customers:
    raise ValueError(f"Customers tabelist puuduvad veerud: {missing_customers}")

# andmete ühendamine
df = pd.merge(
    sales_df,
    customers_df[["customer_id", "first_name", "last_name", "email", "phone", "city"]],
    on="customer_id",
    how="left"
)

# andmete puhastamine

# 1. Kontrolli duplikaate
duplicate_count = df.duplicated(subset=["invoice_id"]).sum()
print(f"\nDuplikaadid invoice_id järgi: {duplicate_count}")
df = df.drop_duplicates(subset=["invoice_id"], keep="first")

# 2. Kontrolli NULL väärtusi
print(f"\nNULL väärtused enne puhastust:\n{df.isnull().sum()}")

# customer_id peab olemas olema
df = df.dropna(subset=["customer_id"])

# jätame alles kliendid, kellel on kas email
# või nimi + telefon
df = df[
    (df["email"].notna())
    |
    (
        df["first_name"].notna()
        & df["last_name"].notna()
        & df["phone"].notna()
    )
].copy()

# 3. Teisenda kuupäevad
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")
df = df.dropna(subset=["sale_date"]).copy()

# 4. Eemalda ebareaalistlikud andmed
today = pd.to_datetime("2025-03-05")
df = df[df["sale_date"] <= today].copy()

# total_price peab olema number
df["total_price"] = pd.to_numeric(df["total_price"], errors="coerce")
df = df.dropna(subset=["total_price"]).copy()

negative_count = (df["total_price"] < 0).sum()
zero_count = (df["total_price"] == 0).sum()

print("\nNegatiivsed summad:", negative_count)
print("Nullsummad:", zero_count)

df = df[df["total_price"] > 0].copy()

# Puhastusraport
print(f"\nPuhastatud andmestik: {df.shape[0]} rida, {df.shape[1]} veergu")
print(f"Unikaalseid kliente: {df['customer_id'].nunique()}")
print(f"Kuupäevavahemik: {df['sale_date'].min()} kuni {df['sale_date'].max()}")

# Graafik 1: müük läbi aegade
sales_over_time = (
    df.groupby(df["sale_date"].dt.date)["total_price"]
    .sum()
    .reset_index()
)

sales_over_time.columns = ["Kuupäev", "Müük"]

fig_line = px.line(
    sales_over_time,
    x="Kuupäev",
    y="Müük",
    title="Müük läbi aegade",
    markers=True
)

fig_line.update_layout(
    xaxis_title="Kuupäev",
    yaxis_title="Müük (€)"
)

fig_line.show()

# Graafik 2: müük kuude lõikes
monthly_sales = (
    df.groupby(df["sale_date"].dt.to_period("M"))["total_price"]
    .sum()
    .reset_index()
)

monthly_sales["sale_date"] = monthly_sales["sale_date"].astype(str)
monthly_sales.columns = ["Kuu", "Müük"]

fig_monthly = px.bar(
    monthly_sales,
    x="Kuu",
    y="Müük",
    title="Müük kuude lõikes",
    text_auto=".2s"
)

fig_monthly.update_layout(
    xaxis_title="Kuu",
    yaxis_title="Müük (€)"
)

fig_monthly.show()

# RFM analüüs

# 1. Recency
recency = df.groupby("customer_id")["sale_date"].max().reset_index()
recency.columns = ["customer_id", "last_purchase_date"]
recency["recency_days"] = (today - recency["last_purchase_date"]).dt.days

# 2. Frequency
frequency = df.groupby("customer_id")["sale_id"].nunique().reset_index()
frequency.columns = ["customer_id", "frequency"]

# 3. Monetary
monetary = df.groupby("customer_id")["total_price"].sum().reset_index()
monetary.columns = ["customer_id", "monetary_value"]

# Ühenda RFM tabel
rfm = (
    recency[["customer_id", "recency_days"]]
    .merge(frequency, on="customer_id")
    .merge(monetary, on="customer_id")
)

# RFM skoorid
rfm["R_score"] = safe_qcut(
    rfm["recency_days"],
    q=5,
    labels=[5, 4, 3, 2, 1],
    reverse=True
).astype(int)

rfm["F_score"] = safe_qcut(
    rfm["frequency"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_score"] = safe_qcut(
    rfm["monetary_value"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Koguskoor
rfm["RFM_Score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

# RFM segmentide määratlemine
def segment_customer(row):
    if row["RFM_Score"] >= 13:
        return "VIP Champions"
    elif row["RFM_Score"] >= 10:
        return "Loyal Customers"
    elif row["RFM_Score"] >= 7:
        return "Potential Loyalists"
    elif row["RFM_Score"] >= 4:
        return "At Risk"
    else:
        return "Lost"


rfm["Segment"] = rfm.apply(segment_customer, axis=1)

# Segmentide suurus
print("\nSegmendid:")
print(rfm["Segment"].value_counts())

vip_count = (rfm["Segment"] == "VIP Champions").sum()
vip_revenue = rfm.loc[rfm["Segment"] == "VIP Champions", "monetary_value"].sum()
total_revenue = rfm["monetary_value"].sum()
vip_share = (vip_revenue / total_revenue * 100) if total_revenue > 0 else 0

print(f"\nVIP Champions: {vip_count} klienti")
print(f"VIP-de tulu: EUR {vip_revenue:,.2f}")
print(f"VIP-de osakaal tulust: {vip_share:.2f}%")

# Graafik 3: RFM segmentide jaotus
segment_counts = rfm["Segment"].value_counts().reset_index()
segment_counts.columns = ["Segment", "Klientide arv"]

fig_pie = px.pie(
    segment_counts,
    names="Segment",
    values="Klientide arv",
    title="RFM segmentide jaotus klientide arvu järgi"
)

fig_pie.update_traces(textposition="inside", textinfo="percent+label")
fig_pie.show()

# Graafik 4: RFM hajuvusdiagramm
fig_scatter = px.scatter(
    rfm,
    x="recency_days",
    y="monetary_value",
    color="Segment",
    size="frequency",
    hover_data=["customer_id", "R_score", "F_score", "M_score", "RFM_Score"],
    title="RFM hajuvusdiagramm: Recency vs Monetary"
)

fig_scatter.update_layout(
    xaxis_title="Recency ehk päevi viimasest ostust",
    yaxis_title="Monetary ehk kogukulutus (€)"
)

fig_scatter.show()

# TOP 10 VIP kliendid
customer_info = (
    df[["customer_id", "first_name", "last_name", "email", "phone", "city"]]
    .drop_duplicates(subset=["customer_id"])
)

rfm_with_names = rfm.merge(customer_info, on="customer_id", how="left")

rfm_with_names["customer_name"] = (
    rfm_with_names["first_name"].fillna("") + " " +
    rfm_with_names["last_name"].fillna("")
).str.strip()

rfm_with_names["customer_label"] = rfm_with_names.apply(
    lambda row: row["customer_name"] if row["customer_name"] else str(row["customer_id"]),
    axis=1
)

top_10_vip = (
    rfm_with_names[rfm_with_names["Segment"] == "VIP Champions"]
    .sort_values("monetary_value", ascending=False)
    .head(10)
)

if not top_10_vip.empty:
    fig_top_vip = px.bar(
        top_10_vip,
        x="customer_label",
        y="monetary_value",
        hover_data=["customer_id", "email", "phone", "frequency", "recency_days"],
        title="TOP 10 VIP klienti kogukulutuse järgi",
        text_auto=".2s"
    )

    fig_top_vip.update_layout(
        xaxis_title="Klient",
        yaxis_title="Kogukulutus (€)",
        xaxis_tickangle=-45
    )

    fig_top_vip.show()
else:
    print("\nVIP kliente ei leitud TOP 10 graafiku jaoks.")

# Lisa customer info lõplikku RFM tabelisse ainult ühe korra
rfm_export = rfm.merge(customer_info, on="customer_id", how="left")

# Ekspordi kogu RFM tabel
rfm_export.to_csv("output/rfm-analysis.csv", index=False)

# Ekspordi ainult VIP-d
vip = rfm_export[rfm_export["Segment"] == "VIP Champions"]
vip.to_csv("output/vip-customers.csv", index=False)

# Ekspordi At Risk
at_risk = rfm_export[rfm_export["Segment"] == "At Risk"]
at_risk.to_csv("output/at-risk-customers.csv", index=False)

print("\nFailid eksporditud kausta output/:")
print("- rfm-analysis.csv")
print("- vip-customers.csv")
print("- at-risk-customers.csv")

Müügiandmete kuju: (10118, 12)
Klientide andmete kuju: (3150, 9)
Müügiandmete veerud: ['id', 'sale_id', 'invoice_id', 'sale_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'total_price', 'channel', 'store_location', 'payment_method']
Klientide andmete veerud: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'registration_date', 'loyalty_tier', 'birth_year']

Duplikaadid invoice_id järgi: 0

NULL väärtused enne puhastust:
id                   0
sale_id              0
invoice_id           0
sale_date            0
customer_id        988
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3462
payment_method       0
first_name         988
last_name          988
email             1944
phone              988
city               988
dtype: int64

Negatiivsed summad: 179
Nullsummad: 0

Puhastatud andmestik: 8923 rida, 17 veergu
Unikaalseid kliente: 2540
Kuupäevavahemik: 2023-01-01 


Segmendid:
Segment
Loyal Customers        970
Potential Loyalists    893
At Risk                408
VIP Champions          211
Lost                    58
Name: count, dtype: int64

VIP Champions: 211 klienti
VIP-de tulu: EUR 383,903.05
VIP-de osakaal tulust: 14.38%



Failid eksporditud kausta output/:
- rfm-analysis.csv
- vip-customers.csv
- at-risk-customers.csv
